# Step 3: Tool Calling (Function Calling)

## What is Tool Calling?

LLMs are trained on static data — they can't check today's weather, do exact math, or call your database on their own. **Tool calling** bridges that gap.

You describe tools to the model using JSON schema. The model decides *when* to call them, generates *structured arguments*, and your code executes the actual function. Then you return the result back to the model so it can form a final answer.

### The 5-Step Flow

```
1. You send:   user message + list of tool definitions
2. Model returns: tool_call (name + JSON args)  ← model DECIDES to use a tool
3. You execute:   run the actual function in Python
4. You send:   result back to model as a tool_result message
5. Model returns: final human-readable answer
```

### How the Model Chooses Tools

The model reads each tool's `description` and `parameters` and matches them against the user's intent. Good descriptions = better tool selection. The `tool_choice` parameter controls behavior:
- `"auto"` — model decides (default)
- `"required"` — model must use at least one tool
- `"none"` — model must not use any tool
- specific tool name — forces that exact tool

---
We'll use **Groq** (same API style as OpenAI) with the free `llama-3.3-70b-versatile` model.


## Setup

**Step 1:** Install the required packages.
Run the cells below one by one in order.

> ⚠️ You only need to install packages once. After that you can skip these cells.


In [ ]:
!pip install groq requests python-dotenv

**Step 2:** Create a `.env` file in the same folder as this notebook and add your Groq API key:

```
GROQ_API_KEY=your_key_here
```

Get a free key at: https://console.groq.com


In [ ]:
import json
import math
import requests
import os
from dotenv import load_dotenv
from groq import Groq

# Load environment variables from .env file
# override=True forces a reload even if the variable was already set
load_dotenv(override=True)

# Groq() automatically reads GROQ_API_KEY from the environment
client = Groq()
MODEL = "llama-3.3-70b-versatile"

print("Setup complete.")
print("Working directory:", os.getcwd())
print(".env found:", os.path.exists(".env"))

# Verify the key loaded correctly (only show first 15 chars for security)
key = os.environ.get("GROQ_API_KEY", "NOT FOUND")
if key == "NOT FOUND":
    print("\n⚠️  WARNING: GROQ_API_KEY not found! Make sure your .env file exists and has the key.")
else:
    print(f"API key loaded: '{key[:15]}...' (length: {len(key)})")


---
## Tool 1: Calculator

LLMs are notoriously bad at exact arithmetic — they often hallucinate wrong answers for large numbers.
This tool fixes that by routing all math operations to Python instead of letting the model guess.

**How it works:**
- We define a Python function `calculator()` that does the actual math.
- We write a **JSON Schema** that describes the function to the LLM (its name, what it does, and what arguments it takes).
- The LLM reads the schema and knows *when* and *how* to call it.


In [ ]:
# ─── PART A: The actual Python function ───────────────────────────────────
# This is the real code that runs on your machine.
# The LLM does NOT run this — your Python kernel does.

def calculator(operation: str, a: float, b: float) -> dict:
    """Perform a basic arithmetic operation."""
    ops = {
        "add":      a + b,
        "subtract": a - b,
        "multiply": a * b,
        "divide":   a / b if b != 0 else "Error: division by zero",
        "power":    a ** b,
        "sqrt":     math.sqrt(a) if a >= 0 else "Error: negative sqrt",
        "modulo":   a % b if b != 0 else "Error: modulo by zero",
    }
    result = ops.get(operation, f"Unknown operation: {operation}")
    return {"operation": operation, "a": a, "b": b, "result": result}


# ─── PART B: JSON Schema — describes the tool to the LLM ──────────────────
# The LLM reads this to understand what the tool can do and what inputs it needs.
# Think of it as a "menu" that tells the model: "here's a tool, here's how to use it."

calculator_tool = {
    "type": "function",
    "function": {
        "name": "calculator",          # Must match the Python function name
        "description": (
            "Performs basic arithmetic on two numbers. "
            "Use this for any math calculation instead of computing yourself."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "operation": {
                    "type": "string",
                    "enum": ["add", "subtract", "multiply", "divide",
                             "power", "sqrt", "modulo"],
                    "description": "The arithmetic operation to perform."
                },
                "a": {"type": "number", "description": "First operand"},
                "b": {"type": "number", "description": "Second operand (ignored for sqrt)"}
            },
            "required": ["operation", "a", "b"]   # LLM must always provide these
        }
    }
}

print("Calculator tool defined.")
print("Quick test:", calculator("multiply", 6, 7))   # Expected: {'result': 42, ...}


### The Tool-Calling Loop

This is the core engine. It handles the full back-and-forth between:
1. Sending your message + tool list to the model
2. Running the tool if the model asks for it
3. Sending the result back so the model can give a final answer

This same loop works for **any** tool — you'll reuse it for weather and multi-tool calls below.


In [ ]:
def run_with_tools(user_message: str, tools: list, tool_map: dict,
                  verbose=True, tool_choice_override: str = "auto"):
    """
    Full tool-calling loop — works for any combination of tools.

    Parameters:
        user_message         : str  — what the user asked
        tools                : list — list of tool JSON schemas (sent to the LLM)
        tool_map             : dict — maps tool name → actual Python function
        verbose              : bool — if True, prints each step so you can follow along
        tool_choice_override : str  — controls tool use:
                                      "auto"     = model decides (default)
                                      "none"     = model must NOT call any tool
                                      "required" = model must call at least one tool

    Returns:
        str — the model's final response

    NOTE: Some Groq models (e.g. llama-3.3-70b-versatile) will try to call
    a built-in 'brave_search' tool for factual questions unless you pass
    tool_choice_override="none". This causes a BadRequestError because
    brave_search is not in your tools list. Always use "none" when you want
    a direct conversational answer.
    """
    messages = [{"role": "user", "content": user_message}]

    if verbose:
        print(f"USER: {user_message}")
        print("-" * 50)

    # ── STEP 1: Send user message + tool definitions to the model ──────────
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice=tool_choice_override   # "auto", "none", or "required"
    )

    msg = response.choices[0].message

    # ── STEP 2: Check if the model wants to call a tool ────────────────────
    # msg.tool_calls is None if the model answered directly, or a list if it
    # wants to call one or more tools. The while-loop handles cases where the
    # model calls tools multiple times (e.g., first calculates, then fetches weather).
    while msg.tool_calls:
        messages.append(msg)  # Keep the assistant's tool-call request in history

        for tc in msg.tool_calls:
            fn_name = tc.function.name               # e.g. "calculator"
            fn_args = json.loads(tc.function.arguments)  # e.g. {"operation": "multiply", "a": 6, "b": 7}

            if verbose:
                print(f"MODEL calls tool: {fn_name}({fn_args})")

            # ── STEP 3: Execute the real Python function ───────────────────
            fn_result = tool_map[fn_name](**fn_args)

            if verbose:
                print(f"TOOL result: {fn_result}")

            # ── STEP 4: Send the result back to the model ──────────────────
            # role="tool" tells the model this message is a tool response.
            # tool_call_id links this result to the specific tool call that triggered it.
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(fn_result)
            })

        # ── STEP 5: Ask the model to generate the final answer ─────────────
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice=tool_choice_override
        )
        msg = response.choices[0].message

    if verbose:
        print(f"\nASSISTANT: {msg.content}")

    return msg.content

print("Loop function ready.")


### Test: Calculator Tool

**Test 1** — A math question the model can't reliably answer on its own.
Watch how it calls the `calculator` tool twice: once for multiplication, once for square root.


In [ ]:
run_with_tools(
    "What is 2847 multiplied by 193, and what's the square root of that result?",
    tools=[calculator_tool],
    tool_map={"calculator": calculator}
)


**Test 2** — A non-math question. The model should answer directly **without** calling any tool.

This shows that `tool_choice="auto"` is smart — the model only uses a tool when it actually needs one.


In [ ]:
# tool_choice="none" tells the model: do NOT call any tool.
# This is needed because some Groq models try to call a built-in
# "brave_search" tool for factual questions — even if it's not in
# your tools list — which causes a BadRequestError.
# By setting tool_choice="none", we force a direct text answer.
run_with_tools(
    "What is the capital of Pakistan?",
    tools=[calculator_tool],
    tool_map={"calculator": calculator},
    tool_choice_override="none"   # added: no tool allowed here
)


> **Key insight**: The model called the calculator for math, but answered the geography question directly.
> This is `tool_choice: "auto"` in action — the model decides when a tool is needed.

---


## Tool 2: Weather (Real API)

This tool makes a **live HTTP request** to [wttr.in](https://wttr.in) — a free, no-API-key-needed weather service.

**Key difference from the calculator:** The calculator uses pure Python math. This tool goes out to the internet and fetches real-time data. The LLM still can't do that on its own — it asks your code to do it.


In [ ]:
# ─── PART A: Python function that calls the real weather API ──────────────

def get_weather(city: str, unit: str = "celsius") -> dict:
    """
    Fetch current weather for a city using wttr.in (no API key needed).

    Parameters:
        city : str  — city name, e.g. "Lahore" or "London, UK"
        unit : str  — "celsius" or "fahrenheit"

    Returns:
        dict — weather data, or {"error": "..."} if the request failed
    """
    try:
        url = f"https://wttr.in/{city}?format=j1"
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()   # raises an exception for 4xx/5xx errors
        data = resp.json()

        current = data["current_condition"][0]
        temp_c  = float(current["temp_C"])
        temp_f  = float(current["temp_F"])
        temp    = temp_c if unit == "celsius" else temp_f
        unit_sym = "°C" if unit == "celsius" else "°F"

        return {
            "city":        city,
            "temperature": f"{temp}{unit_sym}",
            "feels_like":  f"{current['FeelsLikeC']}°C",
            "humidity":    f"{current['humidity']}%",
            "description": current["weatherDesc"][0]["value"],
            "wind_speed":  f"{current['windspeedKmph']} km/h"
        }
    except Exception as e:
        return {"error": str(e), "city": city}


# Quick sanity check — call the function directly (no LLM involved yet)
print("Direct API test:")
print(get_weather("Faisalabad"))


In [ ]:
# ─── PART B: JSON Schema for the weather tool ─────────────────────────────
# This tells the LLM: "there's a tool called get_weather, here's what it does
# and what arguments it accepts."

weather_tool = {
    "type": "function",
    "function": {
        "name": "get_weather",       # Must match the Python function name above
        "description": (
            "Get the current weather for a city. "
            "Returns temperature, humidity, wind speed, and conditions. "
            "Always use this when asked about weather."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "City name, e.g. 'Lahore' or 'London, UK'"
                },
                "unit": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                    "description": "Temperature unit. Default: celsius."
                }
            },
            "required": ["city"]     # unit is optional (has a default value)
        }
    }
}

print("Weather tool defined.")


In [ ]:
# Test: Ask the LLM about weather — it will call get_weather() automatically
run_with_tools(
    "What's the weather like in Karachi right now? Should I carry an umbrella?",
    tools=[weather_tool],
    tool_map={"get_weather": get_weather}
)


---
## Combining Both Tools

Now we pass **both tools** at the same time. The model picks whichever one fits the question — or uses both if needed.

This is how real agents work: a large toolbox, and the model decides what to use.


In [ ]:
# Put both tools and both functions into shared containers
ALL_TOOLS = [calculator_tool, weather_tool]
TOOL_MAP  = {"calculator": calculator, "get_weather": get_weather}

# ── Test 1: Math question — model should use calculator ────────────────────
# "17.5% of 8400" = 8400 * 0.175 = 1470
run_with_tools(
    "What's 17.5% of 8400?",
    tools=ALL_TOOLS, tool_map=TOOL_MAP
)


In [ ]:
# ── Test 2: Weather question — model should use get_weather ───────────────
run_with_tools(
    "How's the weather in Islamabad?",
    tools=ALL_TOOLS, tool_map=TOOL_MAP
)


In [ ]:
# ── Test 3: Question that needs BOTH tools ────────────────────────────────
# The model must:
#   1. Call calculator to average 38 and 24  →  (38+24)/2 = 31
#   2. Call get_weather to fetch live Lahore data
run_with_tools(
    "The high temp in Lahore is 38°C and the low is 24°C. What is the average? Also fetch live weather for Lahore.",
    tools=ALL_TOOLS, tool_map=TOOL_MAP
)


---
## What You Just Learned

| Concept | What it means |
|---|---|
| **Tool definition** | JSON schema — name, description, parameters |
| **Tool selection** | Model reads descriptions, picks based on user intent |
| **Tool call** | Model returns `tool_calls` array instead of plain text |
| **Tool result** | You run the real code and return via `role: "tool"` |
| **Multi-turn** | Model can call tools multiple times in one session |
| **tool_choice** | `auto` / `required` / `none` / specific name |

### Why This Matters for Agents

Tool calling is the core mechanism that turns a passive LLM into an **agent**. Every agent framework (LangChain, LlamaIndex, AutoGen) is built on this exact loop:

```
Think → Choose tool → Execute → Observe result → Think again → ...
```

You've now built that loop from scratch. The next steps (ReAct, agentic loops) are extensions of exactly this pattern.

---
### Common Errors to Watch For

| Error | Likely Cause | Fix |
|---|---|---|
| `AuthenticationError` | API key missing or wrong | Check `.env` file and re-run setup cell |
| `KeyError` in tool_map | Tool name mismatch | Make sure `function.name` in schema matches the dict key in `tool_map` |
| `JSONDecodeError` | LLM returned bad args | Add `try/except` around `json.loads(tc.function.arguments)` |
| `TypeError: unexpected keyword` | LLM passed wrong arg name | Check parameter names in your JSON schema match your Python function |
